## Name: Naveen Gupta

# Practice Exercises: RAG & Vector Databases

**Post-class practice!** In this notebook, you will build your own mini RAG system step by step.

**Instructions:**
- Complete the lines marked with `# TODO` in each exercise
- Run the code and observe the output carefully
- Answer every "Reflection Question" in the markdown cell provided

**Setup:** Run this on Google Colab (Runtime → Change runtime type → GPU recommended, but CPU works too — just slower)


---
## Part 0: Setup

First, install and import the required libraries.


In [ ]:
# Run this first!
!pip install transformers torch sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.3 MB/s eta 0:00:00


In [ ]:
from transformers import pipeline

# Load the text generation model (same as class)
generator = pipeline('text-generation', model='gpt2')

print("Setup complete! ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Setup complete! ✅


---
## Part 1: Hallucination vs RAG — The "Open Book Exam" 📖

Remember: without context, the LLM **guesses** (Hallucination). With RAG, we give it the "book" to read from.


### Exercise 1.1: Catch the Hallucination! 🕵️

**Task:** Ask GPT-2 a question about something it **cannot possibly know** (a made-up or very recent fact). Observe how it confidently makes something up.

Example ideas: *"Who won the 2027 Cricket World Cup?"*, *"What is the name of the mayor of Atlantis?"*


In [ ]:
# TODO: Write a question the model cannot know the answer to
my_question = "Who is going to win the 2026 fifa world cup on monday."   # <-- your impossible question here

res = generator(my_question, max_new_tokens=30)
print(res[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Who is going to win the 2026 fifa world cup on monday. No matter how great the match is, it's going to feel like a second half, and maybe even a third, or even a fourth, or


**🤔 Reflection Question 1.1:** Did the model say "I don't know"? Or did it confidently invent an answer? Why do you think LLMs hallucinate instead of admitting they don't know?

> LLMs may hallucinate because they are trained to predict the most likely next sequence of words rather than determine whether they actually know the correct answer. Their objective is to generate a fluent and plausible response, even when they lack sufficient knowledge. Additionally, if the question involves recent events or information outside the model's training data, the model may still attempt to answer instead of admitting uncertainty, leading to hallucinations.



### Exercise 1.2: Fix It with RAG (Manual Augmentation) 🔧

**Task:** Now fix the hallucination from Exercise 1.1 using the **Augmentation** step:
1. Write a `private_knowledge` string that contains the correct answer to your question
2. Build a `rag_prompt` that combines **Context + Question** (like we did in class)
3. Compare the output with Exercise 1.1


In [ ]:
# TODO: Write the "ground truth" for your question
# Your original question
my_question = "Who is going to win the 2026 FIFA World Cup on Monday?"

# Ground truth (private knowledge)
private_knowledge = (
    "The 2026 FIFA World Cup final will be played between Spain and Argentina on Monday. "
    "The winner is not known yet because the match has not been played."
)

# Build the RAG prompt
rag_prompt = f"""
Context:
{private_knowledge}

Question:
{my_question}

Answer according to the context:
"""

res_rag = generator(rag_prompt, max_new_tokens=30)
print(res_rag[0]["generated_text"])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
The 2026 FIFA World Cup final will be played between Spain and Argentina on Monday. The winner is not known yet because the match has not been played.

Question:
Who is going to win the 2026 FIFA World Cup on Monday?

Answer according to the context:

Spanish coach Jose Maria Garcia (2nd from left) holds on to an empty goal post as the ball is touched by Argentina's David Villar


**🤔 Reflection Question 1.2:** Which of the 3 RAG steps (Retrieval, Augmentation, Generation) did we do **manually** here? Which step is still missing from our system?

> In this exercise, we manually performed the Augmentation step by adding the relevant context (private_knowledge) to the prompt before asking the question. We also manually provided the information that would normally come from Retrieval.

> The step still missing from a real RAG system is automatic Retrieval, where the system searches a knowledge base or external documents to fetch the most relevant context. After retrieval and augmentation, the model performs Generation to produce the final answer.


---
## Part 2: Embeddings — Turning Meaning into Numbers 🔢

Remember: AI understands **Numbers**, not words. Similar meanings → vectors that are **close** on the graph.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model (same as class)
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready! ✅")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready! ✅


### Exercise 2.1: Look Inside an Embedding 👀

**Task:** Convert a sentence into an embedding and inspect it:
1. Encode the sentence `"I love programming"`
2. Print the **shape** (how many numbers?) and the **first 10 numbers**


In [ ]:
sentence = "I love programming"

# Encode the sentence into an embedding
embedding = model.encode(sentence)

print("Shape of embedding:", embedding.shape)
print("First 10 numbers:", embedding[:10])

Shape of embedding: (384,)
First 10 numbers: [-0.03617867 -0.01277372  0.0030063  -0.01690344  0.00948426 -0.06515173
  0.09376633  0.07142349  0.01852631  0.05358271]


**🤔 Reflection Question 2.1:** How many dimensions (numbers) does one embedding have? Can a human read meaning from these numbers directly?

> The embedding has **384 dimensions (numbers)** when using the `all-MiniLM-L6-v2` model. Different embedding models may produce vectors of different sizes.

> Humans cannot directly understand the meaning of these numbers. Individually, the values have no interpretable meaning. Instead, the entire vector collectively represents the semantic meaning of the sentence, allowing similar sentences to have embeddings that are close together in the vector space.



### Exercise 2.2: The King & Queen Test 👑

In class we said: *"King" and "Queen" vectors will be close, but "Apple" will be far away.* Let's **prove it with code**!

**Task:** Compute the similarity between sentence pairs. Higher score = closer meaning.


In [ ]:
from sentence_transformers import util

sentences = [
    "The king rules the country.",      # 0
    "The queen lives in the palace.",   # 1
    "I ate an apple for breakfast."     # 2
]

embeddings = model.encode(sentences)

# Compute cosine similarity between sentence 0 and sentence 1
sim_king_queen = util.cos_sim(embeddings[0], embeddings[1])

# Compute cosine similarity between sentence 0 and sentence 2
sim_king_apple = util.cos_sim(embeddings[0], embeddings[2])

print(f"King vs Queen similarity: {sim_king_queen.item():.4f}")
print(f"King vs Apple similarity: {sim_king_apple.item():.4f}")

King vs Queen similarity: 0.4106
King vs Apple similarity: 0.0614


**🤔 Reflection Question 2.2:** Which pair got the higher similarity score? Does this match the "close on the graph" logic from class?

> The King vs. Queen pair received the higher cosine similarity score because the two sentences are semantically related. In contrast, King vs. Apple had a lower similarity score since they discuss unrelated concepts.

> Yes, this matches the "close on the graph" logic from class. Sentences with similar meanings are represented by embedding vectors that are closer together in the embedding space, resulting in a higher cosine similarity score. Unrelated sentences are farther apart and therefore have a lower similarity score.


---
## Part 3: Build Your Own Vector Database 🗄️

Now the real deal — build a FAISS vector database with **your own documents** and search it.


### Exercise 3.1: Create Your Knowledge Base

**Task:** Write **5 documents of your own** (facts about your college, your city, your favorite topics — anything!). Then:
1. Encode them into vectors
2. Store them in a FAISS index
3. Print how many documents got stored


In [ ]:
import faiss

# TODO: Write your own 5 documents (make them about different topics!)
documents = [
    "",   # <-- doc 1
    "",   # <-- doc 2
    "",   # <-- doc 3
    "",   # <-- doc 4
    "",   # <-- doc 5
]

# TODO: Convert documents into vectors using model.encode()
doc_embeddings = None   # <-- fix this

# TODO: Build the FAISS index
dimension = doc_embeddings.shape[1]        # size of each vector
index = None                               # <-- create faiss.IndexFlatL2(...) here
index.add(np.array(doc_embeddings))        # add vectors to the database

print(f"{index.ntotal} documents have been stored in the Vector Database.")

### Exercise 3.2: Semantic Search Test 🔍

**Task:** Ask a question about one of YOUR documents — but **do NOT use the exact same words** as the document! This tests **Semantic Search** (meaning) vs **Keyword Search** (exact words).

Example: if your document says *"Almonds are healthy for the brain"*, ask *"Which food is good for memory?"*


In [ ]:
# TODO: Write a query that matches one document by MEANING, not exact words
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create your knowledge base
documents = [
    "My college offers undergraduate and postgraduate engineering programs.",
    "The library is open from 8 AM to 8 PM on weekdays.",
    "The computer science department has modern AI and ML laboratories.",
    "Our city is famous for its historical monuments and local cuisine.",
    "Students can participate in coding competitions and hackathons every semester."
]

# Encode documents
document_embeddings = model.encode(documents)

# Create FAISS index
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# Add document embeddings
index.add(np.array(document_embeddings).astype("float32"))

print("Number of documents stored:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of documents stored: 5


**🤔 Reflection Question 3.2:** Did the database find the right document even though you used different words? Explain how this is different from Ctrl+F / keyword search:

> Yes, the database retrieved the correct document even though the query used different words. This is because FAISS searches using embeddings, which capture the semantic meaning of the text, rather than matching exact words.

> This is different from Ctrl+F or keyword search, which only finds documents containing the exact words or phrases entered by the user. Semantic search can retrieve relevant documents even when different words or synonyms are used, making it more flexible and accurate for natural language queries.


### Exercise 3.3: Top-K Retrieval

**Task:** Instead of only the top 1 result, retrieve the **top 3** most similar documents and print them in order with their distances.


In [ ]:
# Write another query
query2 = "Where can students take part in programming contests?"

# Convert the query into an embedding
query_embedding2 = model.encode([query2]).astype("float32")

# Search for the top 3 most similar documents
D, I = index.search(query_embedding2, k=3)

print(f"Question: {query2}\n")

# Print the top 3 results
for rank in range(3):
    print(f"Rank {rank+1}: {documents[I[0][rank]]} (distance: {D[0][rank]:.4f})")

Question: Where can students take part in programming contests?

Rank 1: Students can participate in coding competitions and hackathons every semester. (distance: 0.6047)
Rank 2: My college offers undergraduate and postgraduate engineering programs. (distance: 1.3668)
Rank 3: The computer science department has modern AI and ML laboratories. (distance: 1.3986)


**🤔 Reflection Question 3.3:** Look at the distances of ranks 1, 2, and 3. Is the rank-1 document clearly the best match, or are the distances close? When might retrieving more than 1 document be useful for RAG?

> The rank-1 document had the smallest distance, making it the best semantic match to the query. Depending on the query, the distances for ranks 2 and 3 may be close, indicating that they also contain relevant information.

> Retrieving more than one document is useful in RAG because the answer may require information from multiple sources or documents. Providing the top-k relevant documents gives the language model more context, improving the accuracy, completeness, and reliability of the generated response while reducing the chances of missing important information.


---
## Part 4: 🏆 Final Challenge — Full RAG Pipeline (End-to-End)

Now connect **everything**: Retrieval (Vector DB) → Augmentation (prompt building) → Generation (LLM).

This is the complete flow from class, but built by YOU:

```
User Query → [Vector DB Search] → Best Document → [Attach to Prompt] → [GPT-2] → Answer
```


In [18]:
def my_rag_pipeline(user_query):
    # STEP 1 — RETRIEVAL

    # Encode the user query into an embedding
    q_emb = model.encode([user_query]).astype("float32")

    # Search the FAISS index for the top 1 document
    D, I = index.search(q_emb, k=1)
    retrieved_doc = documents[I[0][0]]

    # STEP 2 — AUGMENTATION

    # Build the RAG prompt
    rag_prompt = f"""
Context:
{retrieved_doc}

Question:
{user_query}

Answer according to the context:
"""

    # STEP 3 — GENERATION

    res = generator(rag_prompt, max_new_tokens=30)

    print("Retrieved Document:", retrieved_doc)
    print("-" * 60)
    print("Final Answer:\n", res[0]["generated_text"])


# Test your pipeline
my_rag_pipeline("What is the city famous for?")

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Document: Our city is famous for its historical monuments and local cuisine.
------------------------------------------------------------
Final Answer:
 
Context:
Our city is famous for its historical monuments and local cuisine.

Question:
What is the city famous for?

Answer according to the context:

The city is known for its historical monuments and local cuisine.

What is your opinion?

The city is renowned for its historical monuments


**🤔 Final Reflection Question:**
1. In your pipeline, what would happen if the Vector DB retrieved the **wrong** document? Would the LLM still give a good answer?
2. Based on this, complete the sentence: *"A RAG system is only as good as its ______."*

> **1.** If the Vector DB retrieves the **wrong document**, the LLM will likely generate an answer based on that incorrect context. Even if the language model is powerful, it cannot reliably produce the correct answer if it is given irrelevant or misleading information. This can result in an incorrect or hallucinated response.

**2.** Complete the sentence:

> **"A RAG system is only as good as its retrieval."**

> This highlights that the quality of the retrieved documents directly affects the quality and accuracy of the final generated answer. If retrieval is accurate, the LLM has the right context to produce a reliable response. If retrieval fails, the final answer is likely to be poor as well.



### Bonus Challenge 4.1: RAG vs Fine-tuning Decision Table 🎓

For each scenario below, decide: **RAG or Fine-tuning?** (Remember: Fine-tuning = 5 years of med school, RAG = textbook in the exam)

| Scenario | RAG / Fine-tuning? | Reason |
|----------|-------------------|--------|
| A company chatbot that must answer from HR policy PDFs that change every month | ? | ? |
| Teaching a model to always respond in Shakespearean English style | ? | ? |
| A news assistant that must know today's headlines | ? | ? |
| A medical model that must deeply understand doctor-style reasoning | ? | ? |


---
| **Scenario**                                                                   | **RAG / Fine-tuning?** | **Reason**                                                                                                                |
| ------------------------------------------------------------------------------ | ---------------------- | ------------------------------------------------------------------------------------------------------------------------- |
| A company chatbot that must answer from HR policy PDFs that change every month | **RAG**                | HR policies change frequently, so retrieving the latest documents is better than retraining the model every time.         |
| Teaching a model to always respond in Shakespearean English style              | **Fine-tuning**        | This changes the model's writing style and behavior, which is best learned through fine-tuning.                           |
| A news assistant that must know today's headlines                              | **RAG**                | News changes constantly, so the model should retrieve the latest articles instead of relying on outdated training data.   |
| A medical model that must deeply understand doctor-style reasoning             | **Fine-tuning**        | Specialized reasoning and domain expertise require the model to learn new patterns, making fine-tuning the better choice. |



### Bonus Challenge 4.2: Break the Retriever! 💥

**Task:** Try to find a query where your Vector DB retrieves the **WRONG** document.

Hints to try:
- A very vague query (e.g., "Tell me something")
- A query about a topic NOT in any of your 5 documents
- A query mixing two topics at once

Print the result and the distance score.


In [19]:
# Try to fool the retriever!
tricky_query = "Who is the principal of the college?"

q_emb = model.encode([tricky_query]).astype("float32")
D, I = index.search(q_emb, k=1)

print(f"Question: {tricky_query}")
print(f"Retrieved: {documents[I[0][0]]}")
print(f"Distance: {D[0][0]:.4f}")

Question: Who is the principal of the college?
Retrieved: My college offers undergraduate and postgraduate engineering programs.
Distance: 1.3727


**🤔 Reflection Question 4.2:** What happened when you asked about a topic that was NOT in your database? Did the retriever say "not found", or did it still return *something*? Why is this a problem for real RAG systems, and how might we solve it? (Hint: think about the distance score!)

> When I asked about a topic that was **not** in my database, the retriever **did not say "not found."** Instead, it returned the document that was **closest in semantic meaning**, even though it was not the correct answer.

> This is a problem for real RAG systems because the language model may use this irrelevant document as context and generate an incorrect or misleading answer. The **distance score** can help identify such cases—if the distance is relatively high (or the similarity is low), it indicates that no document is a good match. A practical solution is to set a **similarity/distance threshold**. If the best retrieved document does not meet the threshold, the system should respond with something like **"I couldn't find relevant information in the knowledge base"** instead of generating an answer from unrelated context.



---
## ✅ Submission Checklist

Before submitting, check:

- [ ] Part 1: Hallucination caught + fixed with manual RAG
- [ ] Part 2: Embedding inspected + King/Queen similarity test done
- [ ] Part 3: Your own 5-document Vector DB built + semantic search + top-3 retrieval
- [ ] Part 4: Full end-to-end RAG pipeline working
- [ ] Bonus 4.1 table filled in
- [ ] All "Reflection Questions" answered
- [ ] **All cells have been run** (outputs are visible)

**Well done! 🎉 You have built a complete RAG system from scratch!**
